In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
from sklearn.cluster import KMeans

def load_data(name):
    df = pd.read_csv(name)
    return df

In [16]:
def data_remplace_Mode_isna(df):
    # Remplacement par la valeur la plus fréquente (Mode)
    for col in ["State", "BankState", "NewExist", "LowDoc","RevLineCr"]:
        df[col] = df[col].fillna(df[col].mode()[0])

In [17]:
def data_remplace_MIS_STATUS(df):
    for col in ["FranchiseCode", "RevLineCr", "LowDoc"]:
        df["MIS_Status"] = df.groupby(col)["MIS_Status"].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "PIF"))


In [18]:
df.head(10)

,Term,FranchiseCode,State,BankState,NAICS_2,NoEmp,NewExist,RetainedJob,CreateJob,UrbanRural,RevLineCr,ApprovalFY,DisbursementGross,GrAppv,LowDoc,MIS_Status,RecessionPeriod,Loan_Usage_Ratio,Employee_Loan_Ratio,JobImpact
0,84,1,IN,OH,45,4,2.0,0,0,0,0.0,1997,60000.0,60000.0,1.0,1.0,0,1.0,0.000067,0
1,60,1,IN,IN,72,2,2.0,0,0,0,0.0,1997,40000.0,40000.0,1.0,1.0,0,1.0,0.000050,0
2,180,1,IN,IN,62,7,1.0,0,0,0,0.0,1997,287000.0,287000.0,0.0,1.0,0,1.0,0.000024,0
3,60,1,OK,OK,0,2,1.0,0,0,0,0.0,1997,35000.0,35000.0,1.0,1.0,0,1.0,0.000057,0
4,240,1,FL,FL,0,14,1.0,7,7,0,0.0,1997,229000.0,229000.0,0.0,1.0,0,1.0,0.000061,14
5,120,1,CT,DE,33,19,1.0,0,0,0,0.0,1997,517000.0,517000.0,0.0,1.0,0,1.0,0.000037,0
6,45,0,NJ,SD,0,45,2.0,0,0,0,0.0,1980,600000.0,600000.0,0.0,0.0,0,1.0,0.000075,0
7,84,1,FL,AL,81,1,2.0,0,0,0,0.0,1997,45000.0,45000.0,1.0,1.0,0,1.0,0.000022,0
8,297,1,FL,FL,72,2,2.0,0,0,0,0.0,1997,305000.0,305000.0,0.0,1.0,0,1.0,0.000007,0
9,84,1,CT,CT,0,3,2.0,0,0,0,0.0,1997,70000.0,70000.0,1.0,1.0,0,1.0,0.000043,0


In [19]:
df=load_data("preparation_data.csv")
# Création des nouvelles variables
recession_years = [2008, 2009, 2020]  # Exemples d'années de récession
df['RecessionPeriod'] = df['ApprovalFY'].apply(lambda x: 1 if x in recession_years else 0)
df['Loan_Usage_Ratio'] = df['DisbursementGross'] / df['GrAppv']
df['Employee_Loan_Ratio'] = df['NoEmp'] / df['GrAppv']
df['JobImpact'] = df['CreateJob'] + df['RetainedJob']

data_remplace_Mode_isna(df)
data_remplace_MIS_STATUS(df)

# Séparation des variables explicatives et de la variable cible
#X = df.drop(columns=["MIS_Status", "State", "BankState"])  # Variables explicatives
#y = df["MIS_Status"]  # Variable cible

# Regrouper les États en clusters avec KMeans
for col in ["State", "BankState"]:
    state_kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)  # 10 clusters (modifiable)
    df[col + "_Cluster"] = state_kmeans.fit_predict(df[[col]])

# Supprimer les colonnes originales State et BankState
X = df.drop(columns=["MIS_Status","State", "BankState"])
y = df["MIS_Status"]  # Variable cible

# Gérer les valeurs manquantes
imputer = SimpleImputer(strategy="most_frequent")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Détection et suppression des outliers avec IsolationForest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(X_imputed)
X_cleaned = X_imputed[outliers == 1]
y_cleaned = y[outliers == 1]

# Diviser les données en train et test
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned)

# Normalisation des données
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Apprendre le modèle
model = LogisticRegression(max_iter=6000)
model.fit(X_train_scaled, y_train)

# Prédiction
y_pred = model.predict(X_test_scaled)

# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


ValueError: could not convert string to float: 'IN'